In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

d:\python\langchain_update\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = ChatOllama(
    model = "llama3.2:latest"
)

In [3]:
class EvalustionSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for essay")
    score : int = Field(description="score out of 10", ge= 0, le= 10)
    

In [4]:
structured_modeloutput= model.with_structured_output(EvalustionSchema)

In [5]:
essay = """ 
The word yoga comes from the Sanskrit word "Yuj," which means union. It is not just physical exercise. It is a complete way of healthy living. Yoga uses body postures called asanas, breathing exercises called pranayama, and meditation to calm the mind.
"""

In [6]:
prompt= f"Evaluate the  language quality of eassy of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
structured_modeloutput.invoke(prompt).score

7

In [18]:
class UPSCState(TypedDict):
    eassy : str
    language_feedback: str
    analysis_feedback : str
    clarity_feedback :str
    overall_feedback : str
    individual_scores : Annotated[list[int], operator.add]
    avg_score = float

In [19]:
def evaluate_language(state: UPSCState):

    prompt = f"Evaluate the  language quality of eassy of the following eassy and provide a feedback and assign a score out of 10 \n {state['eassy']}"
    output = structured_modeloutput.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores' : [output.score]}

In [20]:
def evaluate_analysis(state: UPSCState):

    prompt = f"Evaluate the depth of analysis the following eassy and provide a feedback and assign a score out of 10 \n {state['eassy']}"
    output = structured_modeloutput.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores' : [output.score]}

In [21]:
def evaluate_thought(state: UPSCState):

    prompt = f"Evaluate the clarity of thought of eassy of the following eassy and provide a feedback and assign a score out of 10 \n {state['eassy']}"
    output = structured_modeloutput.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores' : [output.score]}

In [31]:
def final_evaluation(state : UPSCState):

    prompt = f"Based on the following feedbacks create a summary feedback \n language feedback {state['language_feedback']} \n depth of analysis {state['analysis_feedback']} \n and clarity of thought feedback {state['clarity_feedback']} \n"

    final_feedback = model.invoke(prompt).content

    # avg_score
    avg_sccore = sum(state['individual_scores'])/ len(state['individual_scores'])
    print(avg_sccore)

    return {'overall_feedback': final_feedback, 'avg_score': avg_sccore }

In [23]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

graph.compile()
workflow = graph.compile()

In [32]:
workflow = graph.compile()
initial_state = {'eassy' : essay}
final_state = workflow.invoke(initial_state)
print(final_state)

{'eassy': ' \nThe word yoga comes from the Sanskrit word "Yuj," which means union. It is not just physical exercise. It is a complete way of healthy living. Yoga uses body postures called asanas, breathing exercises called pranayama, and meditation to calm the mind.\n', 'language_feedback': 'The essay provides a good introduction to the concept of yoga and its meaning. However, it lacks depth and clarity in its explanation. Here are some suggestions to improve the essay: Use more specific examples of asanas and pranayama to demonstrate their benefits. Provide more context about the history and evolution of yoga. Use transitional phrases to connect ideas between sentences and paragraphs. Provide a clearer conclusion that summarizes the main points. Consider adding a sentence or two to address potential counterarguments. Overall, the essay demonstrates a good understanding of the basics of yoga, but could benefit from more detail and analysis.', 'analysis_feedback': 'The essay provides a

In [25]:
print(final_state['analysis_feedback'])

The essay provides a basic understanding of the word 


In [26]:
print(final_state['language_feedback'])

The essay is well-structured and easy to understand. However, it lacks depth and provides limited information about yoga. The language is simple and straightforward, but it may not be engaging for all readers. The essay could benefit from more specific examples, counterarguments, or personal anecdotes to make it more compelling.


In [27]:
print(final_state['clarity_feedback'])

The clarity of thought in this essay is good, but there are a few areas that need improvement to take it to the next level. Here's a breakdown of the strengths and weaknesses: Strengths: 1. Clear definition of yoga: The essay starts with a clear and concise definition of yoga, setting the stage for the rest of the essay. 2. Logical structure: The essay follows a logical structure, starting with the etymology of yoga and then moving on to its various components. 3. Simple language: The language used is simple and easy to understand, making it accessible to a wide range of readers. Weaknesses: 1. Lack of depth: The essay provides a good overview of yoga, but it lacks depth and detail. 2. No supporting evidence: There is no supporting evidence or examples to back up the claims made in the essay. 3. No conclusion: The essay ends abruptly without a conclusion, leaving the reader with a sense of unfinished business. Score: 6 out of 10.


In [28]:
print(final_state['individual_scores'])

[4, 6, 6]


In [29]:
print(final_state['overall_feedback'])

Here is a summary feedback based on the provided feedbacks:

**Summary Feedback**

The essay has a clear structure and is easy to understand, but it lacks depth and detail, particularly in its exploration of yoga. The language is simple and straightforward, which makes it accessible to a wide range of readers, but may not be engaging enough to hold their attention. The essay could benefit from more specific examples, counterarguments, and personal anecdotes to make it more compelling. Additionally, the clarity of thought is generally good, but there are areas that need improvement, such as the lack of supporting evidence and a conclusion. Overall, the essay scores 6 out of 10, indicating a good foundation but room for improvement in terms of depth, analysis, and engagement.


In [30]:
print(final_state['avg_score'])

KeyError: 'avg_score'